# 01 — Data Audit: Populations, Nesting, Coverage and Coding Priming

MallWorld audit (2026-10) · supersedes the population handling of the archived notebooks in `notebooks/archive/`

---

## Why this notebook exists

Every later MallWorld analysis depends on four decisions made here:
1. **Which posts count as dreams.**
2. **What the unit of analysis is.**
3. **How much of each field is actually coded.**
4. **Which codes the extraction prompt shaped.**

The archived notebooks made these decisions inconsistently and mostly without saying so. This notebook documents each one with numbers. The verified loader `scripts/mallworld_dataset.py` then fixes them for all later notebooks.

## Checks

| # | Check | Why it matters |
|---|---|---|
| D1 | Populations used by the archived notebooks | The thesis (N = 2,678) and four reports (N = 1,926) used different samples |
| D2 | Text, images and length | Short posts, image-only posts and AI images carry different information |
| D3 | Nesting of locations and entities within dreams and authors | Location-level tests treated nested rows as independent |
| D4 | Field coverage | Most fields are "not mentioned"; denominators must be stated |
| D5 | Priming of the extraction prompt | The prompt told the coder what several codes *mean* in the framework |
| D6 | Join keys | `location_id` is unique only within a dream |
| D7 | Duplicate submissions and coder test–retest | Identical texts extracted twice show the coder's consistency |

## D1 — Populations

In [1]:
import sys, json, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import chi2_contingency, spearmanr, fisher_exact
from statsmodels.stats.multitest import multipletests

MW_ROOT = Path.cwd().parent
sys.path.insert(0, str(MW_ROOT / "scripts"))
import mallworld_dataset as mw  # verified loader: schema-checked fields, populations, cluster keys

OUTPUT_DIR = MW_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

posts = mw.load_posts()
primary = posts[posts["primary"]].copy()
print(f"Posts: {len(posts):,} | primary dream reports: {len(primary):,} from {primary['author'].nunique():,} authors")

print()
print("Post types (all structured posts):")
print(posts["post_type"].value_counts().to_string())

ext = posts[posts["has_extractable_dream"]]
print(f"\n'has_extractable_dream' population (archived notebooks 07, 09, 10 and the main thesis): {len(ext):,}")
comp = ext.groupby("post_type").agg(posts=("post_id", "size"), locations=("n_locations", "sum"),
                                    no_text=("has_text", lambda s: int((~s).sum())), with_images=("n_images", lambda s: int((s > 0).sum())),
                                    median_words=("words", "median"))
print(comp.sort_values("posts", ascending=False).to_string())
non_dream = ext[~ext["is_dream_report"]]
print(f"\nNon-dream-report posts inside the thesis population: {len(non_dream):,} ({100 * len(non_dream) / len(ext):.1f}%), "
      f"carrying {non_dream['n_locations'].sum():,} of {ext['n_locations'].sum():,} locations ({100 * non_dream['n_locations'].sum() / ext['n_locations'].sum():.1f}%)")
ai = ext[ext["post_type"] == "ai_visualization"]
print(f"AI visualisations in the thesis population: {len(ai)} (no text: {int((~ai['has_text']).sum())}); their locations were coded from generated images")

dr = posts[posts["is_dream_report"]]
print(f"\nDream reports: {len(dr):,}; with >= 1 location: {(dr['n_locations'] > 0).sum():,} (the 1,926 of archived notebooks 04-06, 11)")
print(f"Exact duplicate submissions among them: {int((dr['is_duplicate'] & (dr['n_locations'] > 0)).sum())}")
print(f"PRIMARY population (dream reports, >= 1 location, duplicates removed): {len(primary):,} dreams, {primary['n_locations'].sum():,} locations")
print("\nArchived notebook 08 filtered on keys that do not exist (ext['post_type'], ext['is_not_dream_report']) and therefore used all "
      f"{len(posts):,} posts, including {int((posts['post_type'] == 'question').sum())} questions and {int((posts['post_type'] == 'media_reference').sum())} media references.")

Posts: 3,732 | primary dream reports: 1,918 from 1,303 authors

Post types (all structured posts):
post_type
dream_report             1792
question                  888
ai_visualization          330
dream_report_with_map     138
meta_discussion           132
theory                    110
media_reference           110
map_only                   66
introduction               41
survey_research            38
shared_dream_claim         34
other                      32
lucid_technique            15
off_topic                   5
spam_troll                  1

'has_extractable_dream' population (archived notebooks 07, 09, 10 and the main thesis): 2,678
                       posts  locations  no_text  with_images  median_words
post_type                                                                  
dream_report            1790       7887        8          251         156.5
question                 453       1400        9           69          88.0
ai_visualization         167        513   

**Finding D1: the main thesis analysed 750 posts that are not dream reports.** The population of archived notebooks 07, 09 and 10 and of the thesis *Correspondential Structure in Collective Dream Space* was every post flagged `has_extractable_dream` (N = 2,678). That set includes:
- 453 questions;
- 167 AI-generated visualisations, 65 of them without any text, whose "locations" and "atmospheres" were coded from generated images;
- 38 map-only posts, 28 introductions and 20 theory posts.

Together these contribute 23.3% of the thesis's locations. The other reports used the 1,926 dream reports with at least one location, so the project ran on two different populations without saying so. Archived notebook 08 filtered on keys that do not exist and analysed all 3,732 posts.

**Decision:** all audited analyses use the **primary population**: dream reports (`dream_report` or `dream_report_with_map`) with at least one location, after removing 8 exact duplicate submissions. That is **N = 1,918 dreams** with 8,685 locations from 1,303 authors.

## D2 — Text, images and narrative length

In [2]:
w = primary["words"]
print(f"Primary dreams: median {w.median():.0f} words (IQR {w.quantile(.25):.0f}–{w.quantile(.75):.0f}); "
      f"no text {int((~primary['has_text']).sum())}; with images {int((primary['n_images'] > 0).sum())} ({100 * (primary['n_images'] > 0).mean():.1f}%)")
print(f"Locations per dream: median {primary['n_locations'].median():.0f}, mean {primary['n_locations'].mean():.2f}, max {primary['n_locations'].max()}")
for col in ["n_locations", "n_connections", "n_interactions", "n_entities"]:
    rho, p = spearmanr(primary["words"], primary[col])
    print(f"  Spearman(words, {col:15s}) = {rho:.2f} (p = {p:.1e})")

L = mw.load_locations()
coded_any = L.groupby("post_id").apply(lambda g: (g[["atmosphere", "light", "cleanliness", "water_presence"]] != "not_mentioned").any(axis=None))
q = pd.qcut(primary.set_index("post_id")["words"], 4, labels=["Q1 shortest", "Q2", "Q3", "Q4 longest"])
share = L.assign(coded=(L["atmosphere"] != "not_mentioned"), q=L["post_id"].map(q)).groupby("q", observed=True)["coded"].mean()
print("\nShare of locations with an atmosphere code, by narrative-length quartile:")
print((100 * share).round(1).to_string())

Primary dreams: median 150 words (IQR 86–277); no text 15; with images 360 (18.8%)
Locations per dream: median 3, mean 4.53, max 34
  Spearman(words, n_locations    ) = 0.55 (p = 1.2e-154)
  Spearman(words, n_connections  ) = 0.51 (p = 4.5e-130)
  Spearman(words, n_interactions ) = 0.68 (p = 1.5e-257)
  Spearman(words, n_entities     ) = 0.51 (p = 3.3e-127)



Share of locations with an atmosphere code, by narrative-length quartile:
q
Q1 shortest    31.1
Q2             39.3
Q3             46.2
Q4 longest     61.9


**Finding D2: posts are short, and length drives how much is coded.** The median dream report has 150 words. Narrative length correlates with the number of locations (ρ = 0.55), interactions (ρ = 0.68) and entities (ρ = 0.51). The share of locations with an atmosphere code rises from 31.1% in the shortest quartile to 61.9% in the longest. As in the NDE project, longer accounts mention more of everything. Co-occurrence analyses must therefore adjust for length.

## D3 — Nesting: locations within dreams, dreams within authors

In [3]:
per_author = primary.groupby("author").size()
print(f"Authors: {len(per_author):,}; single-post authors {100 * (per_author == 1).mean():.1f}%; "
      f"posts by authors with >= 2 posts: {100 * per_author[per_author >= 2].sum() / len(primary):.1f}%; most prolific: {per_author.max()} posts")
print(per_author.value_counts().sort_index().head(8).to_string())

# Intra-dream correlation of the atmosphere valence (one-way random-effects ICC on dreams with >= 2 coded locations)
V = L.dropna(subset=["valence"])
g = V.groupby("post_id")["valence"]
multi = V[V["post_id"].map(g.size()) >= 2]
k = multi.groupby("post_id").size()
grand = multi["valence"].mean()
msb = (k * (multi.groupby("post_id")["valence"].mean() - grand) ** 2).sum() / (len(k) - 1)
msw = ((multi["valence"] - multi.groupby("post_id")["valence"].transform("mean")) ** 2).sum() / (len(multi) - len(k))
k0 = (len(multi) - (k ** 2).sum() / len(multi)) / (len(k) - 1)
icc = (msb - msw) / (msb + (k0 - 1) * msw)
mean_k = V.groupby("post_id").size().mean()
print(f"\nAtmosphere valence: {len(V):,} coded locations in {V['post_id'].nunique():,} dreams (mean {mean_k:.2f} per dream)")
print(f"Intra-dream correlation (ICC1) = {icc:.2f}; design effect = 1 + (m - 1) * ICC = {1 + (mean_k - 1) * icc:.2f}")
print("A chi-square or correlation that treats locations as independent overstates the effective sample by roughly this factor.")

Authors: 1,303; single-post authors 78.7%; posts by authors with >= 2 posts: 46.6%; most prolific: 22 posts
1    1025
2     169
3      54
4      18
5       9
6       5
7       8
8       4

Atmosphere valence: 4,145 coded locations in 1,347 dreams (mean 3.08 per dream)
Intra-dream correlation (ICC1) = 0.31; design effect = 1 + (m - 1) * ICC = 1.64
A chi-square or correlation that treats locations as independent overstates the effective sample by roughly this factor.


**Finding D3: locations are nested, and the archived notebooks ignored the nesting.** Coded atmospheres cluster within dreams (ICC1 = 0.31). With an average of 3.1 coded locations per dream, a location-level test overstates the effective sample size by a design effect of about 1.6, and more for tables built on longer dreams. 78.7% of authors posted once, but authors with two or more posts contribute 46.6% of dreams. The archived notebooks computed every χ² and correlation on locations or entity mentions as if they were independent.

**Decision:** location-level associations are estimated with standard errors clustered on the dream, or with dream-level bootstrap intervals. Within-dream contrasts are used wherever the question allows.

## D4 — Field coverage in the primary population

In [4]:
loc_fields = ["atmosphere", "vertical", "light", "light_temperature", "cleanliness", "water_presence", "privacy_status",
              "reality_stability", "crowding", "state", "affective_response", "somatic_response", "cardinal", "time_of_day"]
cov = []
for f in loc_fields:
    absent = {"not_mentioned", "none"} if f in ("somatic_response",) else {"not_mentioned"}
    if f == "vertical":
        absent = {"not_mentioned", "varies"}
    coded = ~L[f].isin(absent)
    cov.append({"field": f, "coded locations": int(coded.sum()), "% of locations": 100 * coded.mean(),
                "dreams with >= 1 coded": int(L.loc[coded, "post_id"].nunique())})
cov = pd.DataFrame(cov).set_index("field").sort_values("% of locations", ascending=False)
with pd.option_context("display.float_format", "{:.1f}".format):
    print(f"Locations: {len(L):,} in {L['post_id'].nunique():,} dreams")
    print(cov.to_string())
print("\nCardinal directions are coded for", int((L["cardinal"] != "not_mentioned").sum()), "locations:",
      L.loc[L["cardinal"] != "not_mentioned", "cardinal"].value_counts().to_dict())
print("Location types: 'other' =", int((L["location_type"] == "other").sum()), f"({100 * (L['location_type'] == 'other').mean():.1f}%), the largest single type")

Locations: 8,685 in 1,918 dreams
                    coded locations  % of locations  dreams with >= 1 coded
field                                                                      
water_presence                 5103            58.8                    1288
atmosphere                     4293            49.4                    1376
reality_stability              3792            43.7                    1182
affective_response             3770            43.4                    1374
vertical                       2983            34.3                     955
state                          2719            31.3                     965
crowding                       2069            23.8                     995
light                          1591            18.3                     804
time_of_day                    1002            11.5                     449
light_temperature               717             8.3                     431
cleanliness                     689             7.9    

**Finding D4: the "correspondence markers" are coded for a small minority of locations.** The rates below are shares of all 8,685 locations:
- atmosphere 49.4%; vertical level 34.3%; light 18.3%;
- light temperature 8.3%; cleanliness 7.9%; privacy status 4.1%; somatic response 3.0%;
- cardinal direction 66 locations (0.8%).

Every rate an analysis reports is conditional on the marker being coded. Coded locations come from longer, more vivid accounts (D2). The East-West framing of the MallWorld synthesis rests on cardinal directions mentioned in 33 dreams, so it cannot be tested with these data. "Other" is the largest single location type (26.4%), so any analysis in which "other" behaves like a real place is measuring the coder's residual category.

## D5 — What the extraction prompt told the coder

`extract.py` describes the schema as "raw phenomenology, not interpretations". Its system prompt nevertheless tells GPT-5.2 what several markers *mean* in the framework, and it defines some categories partly by the dreamer's affect or by location type. The table quotes the prompt and classifies each instruction.

| Field | Prompt instruction (quoted) | Problem for testing |
|---|---|---|
| `light_temperature` | "Warm/Golden … (Indicates Charity/Good)"; "Cold/White … (Indicates Faith/Truth alone)" | Coder knew the predicted meaning (primed) |
| `somatic_response` | "SOMATIC DISTRESS (Sphere Incompatibility)" | Primed |
| `privacy_status` | "Exposed … (Shameful)", "Crowded Exposure … (Hellish)"; `natural_seclusion` = "OUTDOORS … AND dreamer feels PEACE/COMFORT" | Primed; `natural_seclusion` is **defined by positive affect**, so it cannot be tested against affect |
| `reality_stability` | "(Babylon/State Detector)"; "Plastic/Fake: Luxury that looks like a stage set" | Primed; "plastic" is defined by luxury, so luxury → plastic is circular |
| `vertical` and connection `direction` | "INFER VERTICALITY from the location type … Basement … → DOWN … Roof … → UP" | **Imputed from location type**: vertical effects can be location-type effects |
| `affective_response` | "AFFECTIVE VECTORS (Ruling Love)" | Primed |
| `transit_mode` | "Passive … (Influx)", "Wandering … (World of Spirits)", "Fleeing … (Rejection)" | Primed |
| `authority_nature` | "(Punishing Spirit Detector)" | Primed |
| `entity_type` | `guide` = "Someone helpful", `threat` = "Hostile entity", `watcher` = "Observer entity" | Type is **defined by demeanor**, so type → demeanor is circular |

Priming does not make a code wrong. It does make "the coder found what the framework predicts" weak evidence, because the coder was told what to look for.

In [5]:
vert = L[L["vertical"].isin(list(mw.VERTICAL_LEVEL))]
print(f"Locations with a vertical level: {len(vert):,}")
for t in ["basement", "parking_structure", "underground", "subway", "cave", "attic", "roof"]:
    s = vert[vert["location_type"] == t]["vertical"].value_counts()
    if len(s):
        print(f"  {t:18s} n={s.sum():4d}  " + ", ".join(f"{k} {v}" for k, v in s.items()))
iv = vert["intrinsically_vertical"]
below = vert["vertical_level"] < 0
print(f"\nUnderground-coded locations whose type is intrinsically vertical: {int((below & iv).sum())}/{int(below.sum())} ({100 * (below & iv).sum() / below.sum():.1f}%)")
print(f"All vertical-coded locations that are intrinsically vertical types: {100 * iv.mean():.1f}%")

priv = L[L["privacy_status"] != "not_mentioned"]
bath = priv["location_type"].isin(["bathroom", "bathroom_public", "bathroom_locker_room"])
print(f"\nprivacy_status coded for {len(priv)} locations; bathrooms {int(bath.sum())} ({100 * bath.mean():.1f}%)")
ns = priv[priv["privacy_status"] == "natural_seclusion"]
print(f"natural_seclusion: {len(ns)} locations; affective_response = {ns['affective_response'].value_counts().to_dict()}")

Locations with a vertical level: 2,983
  basement           n=  55  lowest 31, lower 24
  parking_structure  n=  21  ground 6, upper 6, lower 5, lowest 3, uppermost 1
  underground        n=  72  lower 48, lowest 24
  subway             n=  29  lower 26, upper 1, ground 1, lowest 1
  cave               n=  31  lower 22, lowest 5, upper 3, uppermost 1
  attic              n=  19  upper 10, uppermost 9
  roof               n=  33  uppermost 25, upper 8

Underground-coded locations whose type is intrinsically vertical: 192/591 (32.5%)
All vertical-coded locations that are intrinsically vertical types: 10.3%

privacy_status coded for 353 locations; bathrooms 81 (22.9%)
natural_seclusion: 44 locations; affective_response = {'not_mentioned': 22, 'anxiety': 6, 'comfort': 6, 'curiosity': 4, 'delight': 4, 'horror': 1, 'confusion': 1}


**Finding D5: verticality was partly imputed from location type, and several markers were primed.** A third (32.5%) of underground-coded locations are basements, caves, subways, parking structures or generic "underground" locations. The prompt instructed the coder to infer verticality from such types. A "lower = worse atmosphere" effect could therefore be a basement effect. The test must be repeated without intrinsically vertical types and with location type controlled. Privacy status is coded mostly outside bathrooms (77.1%), and `natural_seclusion` was defined in the prompt by the dreamer's peace. Light temperature, somatic distress, reality stability, transit mode and authority nature were each labelled with their predicted framework meaning. Findings on these fields measure, at best, whether the coder applied the framework's reading consistently.

## D6 — Join keys: `location_id` is unique only within a dream

In [6]:
E = mw.load_entities("extractable")
Lx = mw.load_locations("extractable")
print(f"Thesis population: {len(E):,} entity involvements, {len(Lx):,} locations")
print(f"Distinct location_id values across all dreams: {Lx['location_id'].nunique()} (e.g. {sorted(Lx['location_id'].unique())[:5]})")
wrong = E.merge(Lx[["location_id", "vertical"]], on="location_id", how="left")
right = E.merge(Lx[["post_id", "location_id", "vertical"]], on=["post_id", "location_id"], how="left")
old_levels = ["lowest", "lower", "ground", "upper", "uppermost"]
print(f"Join on location_id only (archived notebook 09, Phases 3-4, 8, 12; notebook 04 cell 21): {len(wrong):,} rows, "
      f"{int(wrong['vertical'].isin(old_levels).sum()):,} of them with a vertical level (archived notebook 09 printed 1,851,999)")
print(f"Join on (post_id, location_id): {len(right):,} rows")
print("The wrong join attaches every entity to the same-numbered location in every other dream, so the environment")
print("attached to an entity is effectively random. Results built on it show 'invariance' because the link was destroyed.")

Thesis population: 4,235 entity involvements, 11,351 locations
Distinct location_id values across all dreams: 892 (e.g. ['loc_1', 'loc_10', 'loc_10_bookstore_closed_state', 'loc_10_elevator_interior', 'loc_10_giant_apartment_multi_story_crowded_dark'])


Join on location_id only (archived notebook 09, Phases 3-4, 8, 12; notebook 04 cell 21): 7,158,025 rows, 1,851,999 of them with a vertical level (archived notebook 09 printed 1,851,999)
Join on (post_id, location_id): 4,235 rows
The wrong join attaches every entity to the same-numbered location in every other dream, so the environment
attached to an entity is effectively random. Results built on it show 'invariance' because the link was destroyed.


**Finding D6: the "entity autonomy" results come from a join bug.** `location_id` values (`loc_1`, `loc_2`, …) repeat across dreams; only 892 distinct values exist. Joining the 4,235 entity involvements of the thesis population to locations on `location_id` alone produces 7,158,025 rows, 1,851,999 of them with a vertical level. That is exactly the number archived notebook 09 printed as "Entities with vertical position". Every analysis built on this join compares entities with environments drawn from other people's dreams. Those analyses include:
- notebook 09, Phase 3 "entity autonomy" (guides 0% hostile at every level; authority functions identical at every level);
- notebook 09, Phase 4 entity–atmosphere tables;
- notebook 09, Phase 8 creature/animal–atmosphere tables (its "χ² ≈ 0, p = 1.0" came from a separate binning error; notebook 03, E6);
- notebook 09, Phase 12;
- notebook 04's interaction × location-type table.

Their "invariance across environments" is what a random environment produces. They are withdrawn and re-estimated with the correct key in notebook 03.

## D7 — Duplicate submissions: the coder against itself

In [7]:
dups = posts[posts["text_checksum"].notna() & posts["text_checksum"].duplicated(keep=False) & posts["is_dream_report"]]
Lall = mw.load_locations("all")
rows = []
for cs, grp in dups.groupby("text_checksum"):
    ids = list(grp.sort_values("date")["post_id"])
    for other in ids[1:]:
        a, b = Lall[Lall["post_id"] == ids[0]], Lall[Lall["post_id"] == other]
        ta, tb = set(a["location_type"]), set(b["location_type"])
        aa = set(a.loc[a["atmosphere"] != "not_mentioned", "atmosphere"]); ab = set(b.loc[b["atmosphere"] != "not_mentioned", "atmosphere"])
        jac = lambda x, y: 1.0 if not (x | y) else len(x & y) / len(x | y)
        rows.append({"pair": f"{ids[0]}/{other}", "words": int(grp["words"].iloc[0]), "n_loc": f"{len(a)}/{len(b)}",
                     "type Jaccard": jac(ta, tb), "atmosphere Jaccard": jac(aa, ab),
                     "atmospheres": f"{sorted(aa)} / {sorted(ab)}"})
tr = pd.DataFrame(rows)
with pd.option_context("display.float_format", "{:.2f}".format):
    print(tr.to_string(index=False))
print(f"\nSame number of locations in {sum(x.split('/')[0] == x.split('/')[1] for x in tr['n_loc'])}/{len(tr)} pairs; "
      f"mean location-type Jaccard {tr['type Jaccard'].mean():.2f}; mean atmosphere-set Jaccard {tr['atmosphere Jaccard'].mean():.2f}")

           pair  words n_loc  type Jaccard  atmosphere Jaccard                                                                       atmospheres
1on4wz7/1on4x44     69   6/5          1.00                0.50                                            ['neutral', 'welcoming'] / ['neutral']
1k38s73/1k38sdv     71   2/2          1.00                1.00                               ['eerie', 'threatening'] / ['eerie', 'threatening']
17ahqbz/17ahqcr    241   4/4          1.00                1.00 ['neutral', 'threatening', 'welcoming'] / ['neutral', 'threatening', 'welcoming']
17ahqbz/17ahqcv    241   4/4          1.00                0.50 ['neutral', 'threatening', 'welcoming'] / ['neutral', 'nostalgic', 'threatening']
17c1trq/17c1trr    227   2/2          1.00                1.00                                                         ['chaotic'] / ['chaotic']
  veg2wj/veg3dc     52   3/3          1.00                1.00                                                                    

**Finding D7: on identical texts, the coder agrees on places but less on atmosphere.** Eight accidental double submissions were extracted independently. The location types agree perfectly in every pair, and the number of locations in 7 of 8. The set of atmospheres agrees fully in 5 of 8 pairs (mean Jaccard 0.75). Atmosphere is the outcome of most archived tests, so its coder noise is examined further in notebook 08.

## Summary

In [8]:
summary = {
    "all posts": len(posts),
    "thesis population (has_extractable_dream)": int(posts["has_extractable_dream"].sum()),
    "  of which not dream reports": int((posts["has_extractable_dream"] & ~posts["is_dream_report"]).sum()),
    "primary population (dreams)": len(primary),
    "primary locations": int(primary["n_locations"].sum()),
    "authors": int(primary["author"].nunique()),
    "single-post authors %": round(100 * (primary.groupby("author").size() == 1).mean(), 1),
    "atmosphere ICC1 within dream": round(icc, 2),
    "locations with atmosphere coded %": round(100 * (L["atmosphere"] != "not_mentioned").mean(), 1),
    "locations with vertical level %": round(100 * L["vertical_level"].notna().mean(), 1),
    "underground locations of intrinsically vertical type %": round(100 * (below & iv).sum() / below.sum(), 1),
    "wrong-join rows (thesis)": len(wrong),
}
for k_, v_ in summary.items():
    print(f"{k_:55s} {v_}")

all posts                                               3732
thesis population (has_extractable_dream)               2678
  of which not dream reports                            750
primary population (dreams)                             1918
primary locations                                       8685
authors                                                 1303
single-post authors %                                   78.7
atmosphere ICC1 within dream                            0.31
locations with atmosphere coded %                       49.4
locations with vertical level %                         34.3
underground locations of intrinsically vertical type %  32.5
wrong-join rows (thesis)                                7158025
